<a href="https://colab.research.google.com/github/seyma26n/GROUP38_CENG313/blob/main/notebooks/Assignment1_eda_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1 — Image Matching Challenge 2025

In [ ]:
import os
import sys
import subprocess

REPO_URL = "https://github.com/seyma26n/GROUP38_CENG313.git"
REPO_DIR = "/content/GROUP38_CENG313"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.data import (
    load_labels, select_datasets, get_image_names, build_scene_map,
    valid_pose_format, calculate_camera_center,
    download_dataset_images, flatten_dataset_images, summarize_datasets
)
print("Reusable data helpers imported from src/data.py")

## Drive Connection

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

data_path = "/content/drive/MyDrive/CENG313_Project/data"

labels_path = os.path.join(data_path, "train_labels.csv")

stairs_folder = os.path.join(data_path, "train", "stairs")

haiper_folder = os.path.join(data_path, "train", "imc2023_haiper")

In [ ]:
print("CSV exists:", os.path.isfile(labels_path))

print("Stairs folder exists:", os.path.isdir(stairs_folder))

print("Haiper folder exists:", os.path.isdir(haiper_folder))

## Exploratory Data Analysis (EDA)

Exploratory Data Analysis (EDA) is the process of examining and understanding a dataset before applying machine learning models. Its main purpose is to identify patterns, distributions, and potential data quality issues through statistical analysis and visualization.

In this project, EDA is performed on the Image Matching Challenge 2025 dataset to better understand its structure and characteristics. The analysis consists of seven sections:

1. **Dataset Overview:** Examining the general structure and basic information of the dataset.
2. **Dataset Quality Check:** Checking for missing values, duplicate records, and data inconsistencies.
3. **Image Distribution Analysis:** Analyzing the distribution of images across datasets and scenes.
4. **Scene and Outlier Analysis:** Examining scene diversity and the distribution of outlier images.
5. **Image Resolution Analysis:** Investigating image dimensions, resolutions, and aspect ratios.
6. **Visual Similarity Analysis:** Comparing images from the same and different scenes to identify visual similarities and differences.
7. **Camera Pose Analysis:** Examining rotation and translation data and visualizing camera positions.

These analyses provide a better understanding of the dataset and help identify potential challenges for image matching, scene clustering, and camera pose estimation in the following stages of the project.

### Dataset Overview

The training dataset of the Image Matching Challenge 2025 consists of 13 datasets containing a total of 1,945 images. Across these datasets, 35 unique dataset-scene pairs were identified. Each dataset contains images belonging to one or more scenes. This structure provides a variety of image groups for analyzing scene distributions and developing image matching and clustering methods.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = load_labels(labels_path)

# First 5 row
print("")
display(df.head())


In [ ]:
# number of rows and column
print("Row, column numbers:")
print(df.shape)

In [ ]:
# name of the columns
print("Column names - features: ")
print(df.columns)

In [ ]:
# Data types
print("Data types:")
df.info()

In [ ]:
# Number of total datasets
print("toplam dataset sayısı: ")
print(df["dataset"].nunique())

In [ ]:
# Total number of different dataset-scene pairs
print("dataset-scene çifti sayısı: ")
print(df[["dataset", "scene"]].drop_duplicates().shape[0])
print("---------------------------------------------------------")

### Dataset Quality Check

In this section, the dataset is examined to identify potential data quality issues, such as missing values, duplicate records, and inconsistencies. These checks help ensure that the data is reliable and suitable for further analysis. Since the dataset is already prepared and provided as part of a Kaggle competition, significant data quality issues are not expected. However, basic quality checks are still performed to verify the dataset's consistency and completeness.

In [ ]:
# Missing values
print("eksik değerlerin sayıları: ")
print(df.isnull().sum())

In [ ]:
# Dublicated rows
print("tamamen aynı olan satırların sayısı :")
print(df.duplicated().sum())

In [ ]:
# Repeated image recordings within the same dataset
print(" Aynı dataset içerisinde tekrarlanan görüntü kayıtları: ")
print(df.duplicated(subset=["dataset", "image"]).sum())

### Image Distrubution Analysis

In this section, the distribution of images across datasets and scenes is analyzed. First, the number of images in each dataset and scene is calculated and visualized using bar charts. Then, the scenes with the largest and smallest numbers of images are identified. Finally, a summary table is created to present basic statistics such as mean, median, minimum, maximum, and standard deviation. These analyses help understand how images are distributed and identify possible imbalances between scenes and datasets.

In [ ]:
# Importing the matplotlib library to create the graph.
import matplotlib.pyplot as plt

# Number of images per dataset
print(" Dataset başına düşen görüntü sayısı :")
print(" ")

dataset_counts = df["dataset"].value_counts().reset_index()

dataset_counts.columns = ["Dataset", "Image Count"]

# Table look
#display(dataset_counts)

# Graph look
plt.figure(figsize=(12, 5))

plt.bar(
    dataset_counts["Dataset"],
    dataset_counts["Image Count"]
)

plt.title("Number of Images per Dataset")
plt.xlabel("Dataset")
plt.ylabel("Number of Images")

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()



In [ ]:
# Number of images per scene
print("Scene başına düşen görüntü sayısı")
scene_counts = (
    df.groupby(["dataset", "scene"])
    .size()
    .reset_index(name="Image Count")
)

# Table Look
#display(scene_counts)

# Graph look
# Yer kalabalığını önlemek için en fazla görsel içeren 10 scene ile bir grafik oluşturuyoruz
top_scenes = scene_counts.nlargest(10, "Image Count")

plt.figure(figsize=(12, 5))

plt.bar(
    top_scenes["scene"],
    top_scenes["Image Count"]
)

plt.title("Top 10 Scenes by Image Count")
plt.xlabel("Scene")
plt.ylabel("Number of Images")

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# The largest and the smallest scenes

# The 5 scenes with the most images
display(scene_counts.nlargest(5, "Image Count"))

# 5 scenes with the fewest images
display(scene_counts.nsmallest(5, "Image Count"))

In [ ]:
# Summary table --> Mean, median, min, max values
print(" ")
print("Özet Tablo: ")
counts = scene_counts["Image Count"]

print("Mean:", counts.mean())
print("Median:", counts.median())
print("Minimum:", counts.min())
print("Maximum:", counts.max())
print("Standard Deviation:", counts.std())

### Scene and Outliar Analysis

In this section, the number of scenes and the distribution of outlier images across datasets are analyzed. Since outlier images are labeled as "outliers" in the scene column, they can be identified and separated from the actual scenes. A summary table is created to show the total number of images, the number of scenes, the outlier count, and the outlier percentage for each dataset.

Additionally, a bar chart is used to visualize the number of scenes per dataset. Finally, datasets are classified as single-scene or multi-scene based on their scene counts. These analyses help understand the scene structure of the datasets and the distribution of outlier images, which are important for the image clustering task.



In [ ]:
# First, adding an outlier column to the data for outlier checking.

df["is_outlier"] = df["scene"] == "outliers.

valid_df = df[df["scene"] != "outliers"]

# General information per dataset, table look
summary = df.groupby("dataset").agg(
    Total_Images=("image", "count"),
    Outlier_Count=("is_outlier", "sum")
)

# Adding the actual scene numbers
# Grouping by datasets and counting the different scene labels within each dataset.
summary["Scene_Count"] = (
    valid_df.groupby("dataset")["scene"].nunique()
)

# Calculating the outlier ratio
summary["Outlier_Rate"] = (
    summary["Outlier_Count"] / summary["Total_Images"] * 100
)

# Editing the table
summary = summary.reset_index()

display(summary.round(2))

print(" ")
print(" ")

# Graph look
plt.figure(figsize=(12,5))

plt.bar(summary["dataset"], summary["Scene_Count"])

plt.title("Number of Scenes per Dataset")
plt.xlabel("Dataset")
plt.ylabel("Number of Scenes")

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(" ")
print(" ")

# Comparing single-scene and multi-scene datasets
single_scene = (summary["Scene_Count"] == 1).sum()
multi_scene = (summary["Scene_Count"] > 1).sum()

print("Single-scene datasets:", single_scene)
print("Multi-scene datasets:", multi_scene)



### Image Resolution Analysis

In this section, the image dimensions and resolution distributions are analyzed using selected images from the dataset. The width, height, aspect ratio, and megapixel values of each image are calculated. Summary statistics and histograms are used to examine the distribution of these values. This analysis helps identify differences in image sizes and provides useful information for image preprocessing.

In [ ]:
# Select the datasets used for image resolution analysis
stairs_df = select_datasets(df, ["stairs"])
haiper_df = select_datasets(df, ["imc2023_haiper"])

In [ ]:
#Görüntünün boyutlarını hesaplama

from PIL import Image
import pandas as pd
import os

image_info = []

# Analyze images from the selected datasets
datasets = [
    (stairs_df, stairs_folder, "stairs"),
    (haiper_df, haiper_folder, "imc2023_haiper")
]

for dataset_df, folder, dataset_name in datasets:

    for image_name in dataset_df["image"]:

        image_path = os.path.join(folder, image_name)

        with Image.open(image_path) as img:
            width, height = img.size

        image_info.append({
            "Dataset": dataset_name,
            "Image": image_name,
            "Width": width,
            "Height": height,
            "Aspect Ratio": width / height,
            "Megapixels": (width * height) / 1_000_000
        })

resolution_df = pd.DataFrame(image_info)

display(resolution_df.head())

In [ ]:
# Summary statistics of image dimensions
display(
    resolution_df[
        ["Width", "Height", "Aspect Ratio", "Megapixels"]
    ].describe().round(2)
)

In [ ]:
# Resolution Distribution

plt.figure(figsize=(10, 5))

plt.hist(
    resolution_df["Megapixels"],
    bins=15,
    edgecolor="black"
)

plt.title("Image Resolution Distribution")
plt.xlabel("Resolution (Megapixels)")
plt.ylabel("Number of Images")

plt.tight_layout()
plt.show()

In [ ]:
# Aspect Ratio Distribution

plt.figure(figsize=(10, 5))

plt.hist(
    resolution_df["Aspect Ratio"],
    bins=15,
    edgecolor="black"
)

plt.title("Image Aspect Ratio Distribution")
plt.xlabel("Aspect Ratio (Width / Height)")
plt.ylabel("Number of Images")

plt.tight_layout()
plt.show()

###  Visual Similarity Analysis

In this section, images from the same and different scenes are visually compared. Sample images are selected to examine similarities and differences in viewing angles, backgrounds, and visual features. Outlier images are also examined when available. This analysis helps understand the challenges of image matching and scene clustering.

In [ ]:
# Check the scenes in the Haiper dataset
print(haiper_df["scene"].unique())

In [ ]:
# Number of images per scene
display(haiper_df["scene"].value_counts())

In [ ]:
# Aynı sahneye ait görüntüleri karşılaştırma

# Select 4 images from the same scene
same_scene = haiper_df[
    haiper_df["scene"] == "fountain"
].sample(n=4, random_state=42)

# Create the figure
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (_, row) in zip(axes, same_scene.iterrows()):

    image_path = os.path.join(
        haiper_folder,
        row["image"]
    )

    with Image.open(image_path) as img:
        ax.imshow(img)

    ax.set_title(row["image"], fontsize=9)
    ax.axis("off")

plt.suptitle("Images from the Same Scene: Fountain")
plt.tight_layout()
plt.show()

In [ ]:
# Farklı sahnelere ait görüntülerin karşılaştırılması

# Select 2 images from each scene
different_scenes = (
    haiper_df.groupby("scene", group_keys=False)
    .sample(n=2, random_state=42)
)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, (_, row) in zip(axes.flat, different_scenes.iterrows()):

    image_path = os.path.join(
        haiper_folder,
        row["image"]
    )

    with Image.open(image_path) as img:
        ax.imshow(img)

    ax.set_title(
        f"Scene: {row['scene']}",
        fontsize=11
    )

    ax.axis("off")

plt.suptitle("Visual Comparison Between Different Scenes")
plt.tight_layout()
plt.show()

In [ ]:
# Outliar görüntülerin incelenmesi

outlier_df = df[df["scene"] == "outliers"]

display(
    outlier_df["dataset"].value_counts()
)

### Camera Pose Analysis

Lastly, in this section, the rotation matrices and translation vectors provided in the dataset are examined. First, the camera pose data is checked for missing or invalid values. Then, camera center coordinates are calculated for a selected scene using the rotation and translation information. Finally, these positions are visualized in a 3D scatter plot to examine the spatial distribution of cameras.

In [3]:
import numpy as np
import pandas as pd

# Check missing camera pose values
pose_summary = pd.DataFrame({
    "Total Images": [len(df)],
    "Missing Rotation": [df["rotation_matrix"].isna().sum()],
    "Missing Translation": [df["translation_vector"].isna().sum()]
})

display(pose_summary)

NameError: name 'df' is not defined

In [ ]:
# Check whether a pose field has the expected format
df["Valid_Rotation"] = df["rotation_matrix"].apply(
    lambda x: valid_pose_format(x, 9)
)

df["Valid_Translation"] = df["translation_vector"].apply(
    lambda x: valid_pose_format(x, 3)
)

df["Valid_Pose"] = (
    df["Valid_Rotation"] & df["Valid_Translation"]
)

pose_validation = pd.DataFrame({
    "Status": ["Valid Pose", "Invalid or Missing Pose"],
    "Image Count": [
        df["Valid_Pose"].sum(),
        (~df["Valid_Pose"]).sum()
    ]
})

display(pose_validation)

In [ ]:
# Select images from the Fountain scene
fountain_df = df[
    (df["dataset"] == "imc2023_haiper") &
    (df["scene"] == "fountain") &
    (df["Valid_Pose"])
].copy()


# Calculate camera center from R and T
camera_centers = []

for _, row in fountain_df.iterrows():

    center = calculate_camera_center(
        row["rotation_matrix"],
        row["translation_vector"]
    )

    camera_centers.append({
        "Image": row["image"],
        "X": center[0],
        "Y": center[1],
        "Z": center[2]
    })


camera_df = pd.DataFrame(
    camera_centers,
    columns=["Image", "X", "Y", "Z"]
)

display(camera_df.head())

In [ ]:

if camera_df.empty:
    print("No valid camera poses found for the selected scene.")

else:
    fig = plt.figure(figsize=(10, 7))

    ax = fig.add_subplot(111, projection="3d")

    ax.scatter(
        camera_df["X"],
        camera_df["Y"],
        camera_df["Z"],
        s=35,
        alpha=0.8
    )

    ax.set_title("Camera Center Distribution - Fountain Scene")

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")

    plt.tight_layout()
    plt.show()



## Baseline: Feature-Based Image Clustering

### Objective

The objective of this baseline is to group images belonging to the same scene using local feature matching and graph-based clustering.

### Methodology

The proposed pipeline consists of:

1. **SIFT:** Extract local keypoints and descriptors from images.
2. **FLANN:** Match image features using Lowe's Ratio Test.
3. **Weighted Graph:** Represent images as nodes and feature matches as weighted edges.
4. **Louvain Community Detection:** Identify groups of images belonging to the same scene.
5. **Evaluation:** Measure clustering performance using Adjusted Rand Index (ARI), Pairwise Precision, Recall, and F1-score.

### Datasets

The experiments use two subsets from the Image Matching Challenge 2025 training dataset:

- **stairs:** 51 images, 2 ground-truth scenes.
- **imc2023_haiper:** 54 images, 3 ground-truth scenes.

Ground-truth labels are used only for evaluation, not for clustering.

In [ ]:

# Install the required libraries for the baseline
!pip -q install kaggle opencv-python-headless networkx scikit-learn pandas


In [ ]:
# Import reusable baseline functions after installing their dependencies
from src.models import (
    extract_sift_features, match_sift_features,
    extract_orb_features, match_orb_features
)
from src.models import cluster_images
from src.utils import evaluate_clustering, cluster_composition, plot_method_comparison

print("SIFT, ORB, graph clustering, and evaluation functions imported from src/")


In [ ]:

# Authenticate Kaggle using Google Colab Secrets
import os
from google.colab import userdata

token = userdata.get("KAGGLE_API_TOKEN")
os.environ["KAGGLE_API_TOKEN"] = token

print("Kaggle API token loaded successfully.")


In [ ]:

# Download ground-truth labels from Kaggle
!kaggle competitions download -c image-matching-challenge-2025 -f train_labels.csv -p /content


In [ ]:

# Load the ground-truth labels
import pandas as pd

df = load_labels("/content/train_labels.csv")

print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())

display(df.head())


In [ ]:
# Select the datasets used for baseline evaluation
selected_datasets = ["stairs", "imc2023_haiper"]
baseline_df = select_datasets(df, selected_datasets)

# Number of images and ground-truth scenes per dataset
dataset_summary = summarize_datasets(baseline_df)
display(dataset_summary)

In [ ]:
# Download the Stairs training images using reusable data helpers
stairs_df = select_datasets(baseline_df, ["stairs"])
stairs_image_names = get_image_names(stairs_df)
stairs_folder = download_dataset_images("stairs", stairs_image_names)
flatten_dataset_images(stairs_folder, "stairs", stairs_image_names)
print("Stairs images ready:", len(stairs_image_names))

In [ ]:
# Verify that all Stairs images are in the directory used by feature extraction
assert all(os.path.isfile(os.path.join(stairs_folder, name)) for name in stairs_image_names)
print("Verified Stairs images:", len(stairs_image_names))

In [ ]:
# Download the Haiper training images using reusable data helpers
haiper_df = select_datasets(baseline_df, ["imc2023_haiper"])
haiper_image_names = get_image_names(haiper_df)
haiper_folder = download_dataset_images("imc2023_haiper", haiper_image_names)
flatten_dataset_images(haiper_folder, "imc2023_haiper", haiper_image_names)
print("Haiper images ready:", len(haiper_image_names))

In [ ]:
# Verify that all Haiper images are in the directory used by feature extraction
assert all(os.path.isfile(os.path.join(haiper_folder, name)) for name in haiper_image_names)
print("Verified Haiper images:", len(haiper_image_names))

### SIFT Feature Extraction

We use Scale-Invariant Feature Transform (SIFT) to detect distinctive keypoints and compute local descriptors for each image.

The maximum number of features is set to 1,000. These descriptors are used in the subsequent image-matching stage.

The `extract_sift_features()` function is imported from `src/models.py`.

In [ ]:

# Extract SIFT features from the Stairs dataset

stairs_image_names = get_image_names(stairs_df)
stairs_folder = "/content/train/stairs"

stairs_features = extract_sift_features(
    stairs_image_names,
    stairs_folder,
    nfeatures=1000
)

print("Images processed:", len(stairs_features))

# Show the number of keypoints in the first image
first_image = stairs_image_names[0]
print("First image:", first_image)
print("Keypoints detected:", len(stairs_features[first_image]["keypoints"]))


### FLANN Feature Matching

SIFT descriptors are matched using FLANN with a KD-tree index. Lowe's Ratio Test (threshold = 0.75) is applied to filter ambiguous matches.

For the Stairs dataset, 1,275 image pairs were compared, and 419 pairs had at least 20 good matches. These pairs are candidates for graph edges.

The `match_sift_features()` function is imported from `src/models.py`.

In [ ]:

# Match all image pairs in the Stairs dataset

stairs_pairs_df = match_sift_features(
    stairs_image_names,
    stairs_features,
    ratio=0.75
)

print("Total image pairs:", len(stairs_pairs_df))
print("Pairs with at least 20 good matches:",
      (stairs_pairs_df["good_matches"] >= 20).sum())

display(stairs_pairs_df.head())


### Graph Construction and Louvain Clustering

Each image is represented as a node in an undirected graph. An edge is created between two images when they have at least 20 good SIFT matches. Edge weights correspond to the number of good matches.

We apply weighted Louvain community detection to group visually related images into clusters without using ground-truth scene labels.

The `cluster_images()` function is imported from `src/models.py`.

In [ ]:

# Apply weighted Louvain clustering to the Stairs dataset

stairs_graph, stairs_clusters = cluster_images(
    stairs_image_names,
    stairs_pairs_df,
    min_matches=20,
    resolution=1.0,
    seed=42
)

print("Number of images:", stairs_graph.number_of_nodes())
print("Number of graph edges:", stairs_graph.number_of_edges())
print("Number of clusters:", len(stairs_clusters))
print("Cluster sizes:", sorted(len(cluster) for cluster in stairs_clusters))


### Evaluation Metrics

We evaluate the predicted image clusters against the ground-truth scene labels using the following metrics:

- **Adjusted Rand Index (ARI):** Measures agreement between predicted and true clusters, adjusted for chance.
- **Pairwise Precision:** Measures the proportion of predicted same-scene image pairs that are correct.
- **Pairwise Recall:** Measures the proportion of actual same-scene pairs identified by the model.
- **Pairwise F1-score:** The harmonic mean of pairwise precision and recall.

Ground-truth labels are used only during evaluation.

The `evaluate_clustering()` function is imported from `src/utils.py`.

In [ ]:

# Evaluate weighted Louvain clustering on the Stairs dataset

# Map each image to its ground-truth scene
stairs_scene_map = build_scene_map(stairs_df)

# Calculate evaluation metrics
stairs_metrics = evaluate_clustering(
    stairs_clusters,
    stairs_image_names,
    stairs_scene_map
)

print("Stairs Baseline Evaluation Results")
print("----------------------------------")

for metric, value in stairs_metrics.items():
    print(f"{metric}: {value:.4f}")


In [ ]:

# Prepare the Haiper dataset for baseline evaluation

haiper_image_names = get_image_names(haiper_df)
haiper_folder = "/content/train/imc2023_haiper"

haiper_scene_map = build_scene_map(haiper_df)

print("Haiper images:", len(haiper_image_names))
print("Ground-truth scenes:", haiper_df["scene"].nunique())
print("Image folder:", haiper_folder)


In [ ]:

# Extract SIFT features from the Haiper dataset

haiper_features = extract_sift_features(
    haiper_image_names,
    haiper_folder,
    nfeatures=1000
)

print("Images processed:", len(haiper_features))

# Check keypoints detected in the first image
first_image = haiper_image_names[0]

print("First image:", first_image)
print(
    "Keypoints detected:",
    len(haiper_features[first_image]["keypoints"])
)


In [ ]:

# Match all image pairs in the Haiper dataset

haiper_pairs_df = match_sift_features(
    haiper_image_names,
    haiper_features,
    ratio=0.75
)

print("Total image pairs:", len(haiper_pairs_df))

print(
    "Pairs with at least 20 good matches:",
    (haiper_pairs_df["good_matches"] >= 20).sum()
)

display(haiper_pairs_df.head())


In [ ]:

# Apply weighted Louvain clustering to the Haiper dataset

haiper_graph, haiper_clusters = cluster_images(
    haiper_image_names,
    haiper_pairs_df,
    min_matches=20,
    resolution=1.0,
    seed=42
)

print("Number of images:", haiper_graph.number_of_nodes())
print("Number of graph edges:", haiper_graph.number_of_edges())
print("Number of clusters:", len(haiper_clusters))
print("Cluster sizes:", sorted(len(cluster) for cluster in haiper_clusters))


In [ ]:

# Evaluate weighted Louvain clustering on the Haiper dataset

haiper_metrics = evaluate_clustering(
    haiper_clusters,
    haiper_image_names,
    haiper_scene_map
)

print("Haiper Baseline Evaluation Results")
print("----------------------------------")

for metric, value in haiper_metrics.items():
    print(f"{metric}: {value:.4f}")


## Experimental Results

We evaluated the SIFT + FLANN + Weighted Louvain baseline on two training subsets using the same parameters.

The Stairs dataset contains 51 images from 2 ground-truth scenes, while the Haiper dataset contains 54 images from 3 scenes.

The evaluation compares predicted clusters with ground-truth labels using ARI, Pairwise Precision, Recall, and F1-score.

In [ ]:

# Compare baseline results across both datasets

results_df = pd.DataFrame([
    {
        "Dataset": "Stairs",
        "Images": len(stairs_image_names),
        "True Scenes": stairs_df["scene"].nunique(),
        "Predicted Clusters": len(stairs_clusters),
        **stairs_metrics
    },
    {
        "Dataset": "Haiper",
        "Images": len(haiper_image_names),
        "True Scenes": haiper_df["scene"].nunique(),
        "Predicted Clusters": len(haiper_clusters),
        **haiper_metrics
    }
])

display(results_df)


In [ ]:
# Analyze the composition of predicted Haiper clusters
haiper_composition_df = cluster_composition(haiper_clusters, haiper_scene_map)
display(haiper_composition_df)

## Discussion and Limitations

The SIFT + FLANN + Weighted Louvain baseline achieved an ARI of **0.4559** on Stairs and **0.6650** on Haiper.

On Stairs, the algorithm produced 6 clusters despite the dataset containing only 2 ground-truth scenes, indicating over-segmentation.

On Haiper, the algorithm correctly grouped all 23 Fountain images but merged the 15 Bike and 16 Chairs images into a single cluster. This explains the perfect pairwise recall (1.0000) alongside lower precision (0.6657).

These results highlight a limitation of feature-match-based graph clustering: visually similar images from different scenes can create misleading graph connections.

Potential improvements include geometric verification using RANSAC and more robust edge-selection strategies.

## Conclusion

This baseline demonstrates how SIFT feature extraction, FLANN matching, and weighted Louvain community detection can be combined for unsupervised image clustering.

The method achieved a Pairwise F1-score of **0.6888** on Stairs and **0.7993** on Haiper. However, the results also reveal over-segmentation and incorrect merging of visually related scenes.

Overall, the approach provides a useful reference point for future improvements, particularly through geometric verification and more reliable graph construction.

## Baseline 2 — ORB + BFMatcher + Graph Clustering

### Objective

This baseline uses ORB (Oriented FAST and Rotated BRIEF) to extract local image features and BFMatcher (Brute-Force Matcher) to identify visual correspondences between image pairs.

Images are represented as nodes in a graph, and sufficiently similar image pairs are connected by edges. Weighted Louvain community detection is then applied to the graph to obtain predicted image clusters.

The method is evaluated on the same Stairs and Haiper training subsets used in the SIFT + FLANN + Weighted Louvain baseline, enabling a consistent comparison.

Ground-truth scene labels are used only for evaluation.

The `extract_orb_features()` function is imported from `src/models.py`.

The `match_orb_features()` function is imported from `src/models.py`.

In [ ]:

# Extract ORB features from the Stairs dataset

stairs_orb_features = extract_orb_features(
    stairs_image_names,
    stairs_folder,
    nfeatures=1000
)

print("Images processed:", len(stairs_orb_features))

first_image = stairs_image_names[0]

print("First image:", first_image)
print(
    "Keypoints detected:",
    len(stairs_orb_features[first_image]["keypoints"])
)


In [ ]:

# Extract ORB features from the Haiper dataset

haiper_orb_features = extract_orb_features(
    haiper_image_names,
    haiper_folder,
    nfeatures=1000
)

print("Images processed:", len(haiper_orb_features))

first_image = haiper_image_names[0]

print("First image:", first_image)
print(
    "Keypoints detected:",
    len(haiper_orb_features[first_image]["keypoints"])
)


In [ ]:

# Match ORB features across all Stairs image pairs

stairs_orb_pairs_df = match_orb_features(
    stairs_image_names,
    stairs_orb_features,
    ratio=0.75
)

print("Total image pairs:", len(stairs_orb_pairs_df))

print("\nORB matching statistics:")
display(stairs_orb_pairs_df["good_matches"].describe())

display(stairs_orb_pairs_df.head())


In [ ]:

# Analyze the number of strong ORB matches

print(
    "Pairs with at least 20 good matches:",
    (stairs_orb_pairs_df["good_matches"] >= 20).sum()
)

print("\nTop 10 matching image pairs:")

display(
    stairs_orb_pairs_df.sort_values(
        by="good_matches",
        ascending=False
    ).head(10)
)


In [ ]:

# Match ORB features across all Haiper image pairs

haiper_orb_pairs_df = match_orb_features(
    haiper_image_names,
    haiper_orb_features,
    ratio=0.75
)

print("Total image pairs:", len(haiper_orb_pairs_df))

print("\nORB matching statistics:")
display(haiper_orb_pairs_df["good_matches"].describe())

display(haiper_orb_pairs_df.head())


In [ ]:

# Analyze the number of strong ORB matches

print(
    "Pairs with at least 20 good matches:",
    (haiper_orb_pairs_df["good_matches"] >= 20).sum()
)

print("\nTop 10 matching image pairs:")

display(
    haiper_orb_pairs_df.sort_values(
        by="good_matches",
        ascending=False
    ).head(10)
)


In [ ]:

# Apply Weighted Louvain clustering to Stairs using ORB matches

stairs_orb_graph, stairs_orb_clusters = cluster_images(
    stairs_image_names,
    stairs_orb_pairs_df,
    min_matches=20,
    resolution=1.0,
    seed=42
)

print("Number of images:", stairs_orb_graph.number_of_nodes())
print("Number of graph edges:", stairs_orb_graph.number_of_edges())
print("Number of clusters:", len(stairs_orb_clusters))
print(
    "Cluster sizes:",
    sorted([len(cluster) for cluster in stairs_orb_clusters])
)


In [ ]:

# Apply Weighted Louvain clustering to Haiper using ORB matches

haiper_orb_graph, haiper_orb_clusters = cluster_images(
    haiper_image_names,
    haiper_orb_pairs_df,
    min_matches=20,
    resolution=1.0,
    seed=42
)

print("Number of images:", haiper_orb_graph.number_of_nodes())
print("Number of graph edges:", haiper_orb_graph.number_of_edges())
print("Number of clusters:", len(haiper_orb_clusters))
print(
    "Cluster sizes:",
    sorted([len(cluster) for cluster in haiper_orb_clusters])
)


In [ ]:

# Evaluate ORB clustering on the Haiper dataset

haiper_orb_metrics = evaluate_clustering(
    haiper_orb_clusters,
    haiper_image_names,
    haiper_scene_map
)

print("Haiper ORB Evaluation Results")
print("----------------------------------")
print("ARI:", round(haiper_orb_metrics["ARI"], 4))
print("Precision:", round(haiper_orb_metrics["Precision"], 4))
print("Recall:", round(haiper_orb_metrics["Recall"], 4))
print("Pairwise F1:", round(haiper_orb_metrics["Pairwise F1"], 4))


## SIFT vs ORB Baseline Comparison

We compare SIFT and ORB feature extraction methods using the same Weighted Louvain clustering algorithm on both datasets.

The results show that SIFT achieves higher ARI and Pairwise F1 scores than ORB on both datasets. ORB produces high precision but relatively low recall, indicating that it misses many images belonging to the same scene.

Overall, SIFT provides a stronger baseline for image clustering in this experiment.

In [ ]:
# Evaluate ORB clustering on the Stairs dataset first to define stairs_orb_metrics
stairs_orb_metrics = evaluate_clustering(
    stairs_orb_clusters,
    stairs_image_names,
    stairs_scene_map
)

# Compare SIFT and ORB baseline results
comparison_df = pd.DataFrame([
    {
        "Dataset": "Stairs",
        "Method": "SIFT",
        **stairs_metrics
    },
    {
        "Dataset": "Stairs",
        "Method": "ORB",
        **stairs_orb_metrics
    },
    {
        "Dataset": "Haiper",
        "Method": "SIFT",
        **haiper_metrics
    },
    {
        "Dataset": "Haiper",
        "Method": "ORB",
        **haiper_orb_metrics
    }
])

display(comparison_df.round(4))

In [ ]:
# Plot the baseline comparison using the reusable helper
plot_method_comparison(comparison_df)

### Baseline Results: Graphical Comparison

The graphs show that SIFT outperforms ORB on both datasets in terms of ARI and Pairwise F1-score.

SIFT achieves an ARI of 0.4559 on Stairs and 0.6650 on Haiper, while ORB achieves 0.2329 and 0.1835, respectively.

Similarly, SIFT achieves higher Pairwise F1-scores on both datasets. These results suggest that SIFT provides more reliable feature matching for the image clustering task in our experiments.

Although ORB achieves perfect precision on Haiper, its low recall indicates that it misses many image pairs belonging to the same scene.